# Layer 20 KL regularization controls

Compare the mean and variance terms of the Gaussian KL penalty on frozen AraBEST Layer 20. At 1,000 labels, the notebook fits seven stochastic heads per draw across seeds 490–492. Classifier evaluation uses the posterior mean.

### Kaggle setup

1. Enable Internet and select a GPU. Attach the compatible AraBEST pooled feature cache as an input. The notebook uses the pinned reference checkout for feature loading and data manifests.
2. For exact historical replay, attach a JSON file named `layer20_train_positions.json` with the ordered training-manifest positions keyed by seed. The repository README shows how to export it from the original Layer 20 notebook. Historical memberships are supplied locally.
3. To use fresh balanced draws instead, set `DRAW_NEW_COHORTS = True`. Their training features may require extraction. Set `ENCODE_MISSING = True` if needed and configure `HF_TOKEN` in Kaggle Secrets if dataset access requires it.
4. Run all cells. For a cache-only coverage check, set `PREFLIGHT_ONLY = True`. The export archive contains paired scores, KL components, per-dialect metrics, predictions, and checkpoints.

ADI-20 dev200 is source development. Full Casablanca validation is external development. The latter has 6,819 manifest rows and includes one known audio failure.

## Runtime settings

In [ ]:
from pathlib import Path

# Point this at a specific dataset folder to limit the search, or leave the default.
CACHE_INPUT = Path("/kaggle/input")
# Leave None to find layer20_train_positions.json in the attached inputs.
TRAIN_ASSIGNMENTS = None
DRAW_NEW_COHORTS = False
ENCODE_MISSING = False
PREFLIGHT_ONLY = False
ENCODE_BATCH = 4
MAX_EXTRACTION_HOURS = 6

WORK = Path("/kaggle/working")
CACHE_ROOT = WORK / "adi20_research_cache"
CACHE_ZIP_NAME = "adi20_research_cache.zip"
REPO = Path("/tmp/adi20-e11-reference")
REPO_URL = "https://github.com/alizbeeb5/Arabic-Dialect-Identification.git"
COMMIT = "b64437a37e7eda2f1721a56860278640bf085bdd"
SEEDS = (490, 491, 492)
LATENT_DIM = 128
UPDATES = 1000
BATCH_SIZE = 256
TAP = "conformer_20"

In [ ]:
import hashlib
import importlib.util
import io
import json
import os
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
from zipfile import ZipFile

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from torch import nn
from torch.nn import functional as F

def install_missing(packages):
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def load_reference():
    REPO.mkdir(parents=True, exist_ok=True)

    def git(*args, **kwargs):
        return subprocess.run(["git", "-C", str(REPO), *args], **kwargs)

    if not (REPO / ".git").exists():
        git("init", "--quiet", check=True)
    if git("cat-file", "-e", f"{COMMIT}^{{commit}}", stderr=subprocess.DEVNULL).returncode:
        git("fetch", "--quiet", "--depth=1", REPO_URL, COMMIT, check=True)
    git("checkout", "--quiet", "--detach", COMMIT, check=True)
    sys.path.insert(0, str(REPO))


load_reference()
install_missing({"soundfile": "soundfile"})

from src.experiments import acoustic_classifier as clf, layer_mixture as lm
from src.models import acoustic_encoder as ae

torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ENCODER_DEVICE = torch.device("cuda:1") if torch.cuda.device_count() >= 2 else DEVICE

CONFIG = json.loads((REPO / "configs/layer_mixture.json").read_text())
LABELS = json.loads((REPO / "configs/labels.json").read_text())["label_order"]
LABEL_ID = {label: i for i, label in enumerate(LABELS)}
SHARED = sorted(LABEL_ID[label] for label in ("ALG", "EGY", "JOR", "MAU", "MOR", "PAL", "UAE", "YEM"))
FEATURE_CONFIG = clf.feature_config(CONFIG["encoder"])
PROVENANCE = json.loads(json.dumps(lm.expected_provenance(FEATURE_CONFIG)))
TAP_INDEX = PROVENANCE["layer_names"].index(TAP)
WIDTH = PROVENANCE["output_dim"]
TAG = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
OUT = WORK / f"adi20_icml_kl_controls_{TAG}"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Head training: {DEVICE} | AraBEST extraction if needed: {ENCODER_DEVICE}")

## Load and validate training and evaluation cohorts

In [ ]:
ORIGINAL_TRAIN_DIGESTS = {490: 'b32f689df4bda1192cee783d66f0a027b42db62f7e4dbfb2cc30692ce49c1fd0', 491: '89964792c6ef6a861fee41898f64718135ba432da8f063551b9737fa856fe18f', 492: '6e034fe90fa5131411ca0dc37460358a178c2828afa0b080447bd9dfaec05be0'}
ORIGINAL_EVAL_DIGESTS = {'ADI20_dev200': 'ab77a78422db2280b77797cbd4e5dce54cbbe7746f94d52d3dd01f3428f0209a', 'Casablanca_full6819': '11f1d20094ddf0c88338aa45549961be8d2ff53252f89f7285991c0830488790'}
EXPECTED_SCORED = {'ADI20_dev200': 200, 'Casablanca_full6819': 6818}

In [ ]:
def clip_digest(rows):
    return hashlib.sha256("\n".join(row["clip_id"] for row in rows).encode()).hexdigest()


def read_manifest(path, expected_sha256):
    blob = path.read_bytes()
    if hashlib.sha256(blob).hexdigest() != expected_sha256:
        raise ValueError(f"Manifest checksum mismatch: {path.name}")
    return [json.loads(line) for line in blob.splitlines() if line]


def targets(rows):
    return np.array([LABEL_ID[row["label"]] for row in rows], dtype=np.int64)


def usable(row):
    return row.get("preparation_status", "ok") == "ok"


TRAIN = read_manifest(REPO / "src/data/manifests/train.jsonl",
                      "56a75aa929e2a2c4592012954386cb3d91ff7f256a7004ebe7c4a776573ea4ba")
if DRAW_NEW_COHORTS:
    by_label = defaultdict(list)
    for position, row in enumerate(TRAIN):
        if usable(row) and row["split"] == "train":
            by_label[row["label"]].append(position)
    train_positions = {}
    for seed in SEEDS:
        rng = random.Random(seed)
        positions = [position for label in LABELS for position in rng.sample(by_label[label], 50)]
        rng.shuffle(positions)
        train_positions[str(seed)] = positions
else:
    if TRAIN_ASSIGNMENTS is None:
        candidates = sorted(CACHE_INPUT.rglob("layer20_train_positions.json"))
        if len(candidates) != 1:
            raise FileNotFoundError(
                "Attach one layer20_train_positions.json, set TRAIN_ASSIGNMENTS, or set DRAW_NEW_COHORTS=True."
            )
        assignments_path = candidates[0]
    else:
        assignments_path = Path(TRAIN_ASSIGNMENTS)
    train_positions = json.loads(assignments_path.read_text())

if not isinstance(train_positions, dict) or set(train_positions) != {str(seed) for seed in SEEDS}:
    raise ValueError("Training assignments must have the three configured seed keys.")

TRAIN_BY_SEED = {}
TRAIN_DIGESTS = {}
for seed in SEEDS:
    positions = train_positions[str(seed)]
    if (not isinstance(positions, list) or len(positions) != 1000
            or not all(type(p) is int and 0 <= p < len(TRAIN) for p in positions)):
        raise ValueError(f"Expected 1,000 valid manifest positions for seed {seed}.")
    rows = [TRAIN[position] for position in positions]
    if (len({row["clip_id"] for row in rows}) != 1000
            or Counter(targets(rows)) != dict.fromkeys(range(len(LABELS)), 50)
            or not all(usable(row) and row["split"] == "train" for row in rows)):
        raise ValueError(f"Incorrect training cohort size, balance or membership for seed {seed}.")
    digest = clip_digest(rows)
    if not DRAW_NEW_COHORTS and digest != ORIGINAL_TRAIN_DIGESTS[seed]:
        raise ValueError(f"Historical E11 training assignment mismatch for seed {seed}.")
    TRAIN_BY_SEED[seed] = rows
    TRAIN_DIGESTS[seed] = digest
del TRAIN

MANIFESTS = REPO / "data/manifests"
EVAL = {
    "ADI20_dev200": read_manifest(MANIFESTS / "dev_pilot.jsonl",
                                  "fb8684c35ef68e25834d5a4c82cddbd31044a899a63b665c41d3f9a179052d02"),
    "Casablanca_full6819": read_manifest(MANIFESTS / "casablanca_validation.jsonl",
                                         "5ed7a36491ba6a41782302e1f3d31c3033b4d72ac18f1f7a2d04e85177dd064e"),
}
for split, rows in EVAL.items():
    if clip_digest(rows) != ORIGINAL_EVAL_DIGESTS[split]:
        raise ValueError(f"Historical E11 evaluation assignment mismatch for {split}.")
    if sum(map(usable, rows)) != EXPECTED_SCORED[split]:
        raise ValueError(f"Historical E11 evaluation coverage mismatch for {split}.")
TRAIN_UNION = list({row["clip_id"]: row for rows in TRAIN_BY_SEED.values() for row in rows}.values())
REQUIRED = {row["clip_id"]: row for rows in (TRAIN_UNION, *EVAL.values()) for row in rows if usable(row)}

## Find and check pooled AraBEST features

In [ ]:
# These are the same pooled-feature cache identities used in E11.
CACHE_SPEC = {"encoder": PROVENANCE, "mode": "mean", "frames": False, "format": 1}
CACHE_ID = hashlib.sha256(ae.canonical_json(CACHE_SPEC)).hexdigest()[:20]
LEGACY_ID = PROVENANCE["feature_config_sha256"][:16]
ACTIVE_CACHE = CACHE_ROOT / "mean" / CACHE_ID
CACHE_KEYS = {
    clip_id: ae.clip_cache_key(clip_id, row["audio_sha256"], PROVENANCE)
    for clip_id, row in REQUIRED.items()
}
CACHE_INDEX = {}
CACHE_ZIP_MOUNT = None


def discover_cache():
    global CACHE_ZIP_MOUNT
    roots = [CACHE_ROOT, CACHE_INPUT]
    archives = sorted(CACHE_INPUT.rglob(CACHE_ZIP_NAME)) if CACHE_INPUT.is_dir() else []
    if archives:
        CACHE_ZIP_MOUNT = TemporaryDirectory(prefix="adi20-mounted-cache-")
        for i, path in enumerate(archives):
            dest = Path(CACHE_ZIP_MOUNT.name) / str(i)
            print(f"Opening cache archive {path.name}", flush=True)
            with ZipFile(path) as archive:
                for entry in archive.infolist():
                    member = Path(entry.filename)
                    if member.is_absolute() or ".." in member.parts:
                        raise ValueError(f"Unexpected cache archive member: {entry.filename}")
                    if member.parent.name not in {CACHE_ID, LEGACY_ID}:
                        continue
                    if (member.name in {"config.json", "index.jsonl"}
                            or (member.name.startswith("part-")
                                and member.name.endswith((".npz", ".index.jsonl")))):
                        archive.extract(entry, dest)
            roots.append(dest)

    for root in roots:
        for config in sorted(root.rglob("config.json")):
            directory = config.parent
            if directory.name not in {CACHE_ID, LEGACY_ID}:
                continue
            saved = json.loads(config.read_text())
            if directory.name == LEGACY_ID:
                if any(saved.get(key) != PROVENANCE[key] for key in ae.FeatureCache.IDENTITY_FIELDS):
                    continue
            elif saved != CACHE_SPEC:
                continue

            parts = {path.name for path in directory.glob("part-*.npz")}
            for manifest in [directory / "index.jsonl", *sorted(directory.glob("part-*.index.jsonl"))]:
                if not manifest.is_file():
                    continue
                with manifest.open() as stream:
                    for line in stream:
                        item = json.loads(line)
                        clip_id = item["clip_id"]
                        if (CACHE_KEYS.get(clip_id) == item["cache_key"]
                                and item["part"] in parts):
                            CACHE_INDEX.setdefault(clip_id, (directory / item["part"], item["row"]))


def cache_report():
    sets = {"ADI-20 training (union of three 1K draws)": TRAIN_UNION, **EVAL}
    table = pd.DataFrame([
        {"cohort": name, "rows": len(rows),
         "cached": sum(row["clip_id"] in CACHE_INDEX for row in rows),
         "to_encode": sum(usable(row) and row["clip_id"] not in CACHE_INDEX for row in rows),
         "manifest_failures": sum(not usable(row) for row in rows)}
        for name, rows in sets.items()
    ])
    table.to_csv(OUT / "cache_coverage.csv", index=False)
    display(table)


def write_cache(rows, values, seconds):
    ids = [row["clip_id"] for row in rows]
    keys = [CACHE_KEYS[clip_id] for clip_id in ids]
    stem = "part-" + hashlib.sha256("".join(keys).encode()).hexdigest()[:24]
    part = ACTIVE_CACHE / (stem + ".npz")
    index = ACTIVE_CACHE / (stem + ".index.jsonl")
    staged_part = part.with_suffix(".tmp")
    staged_index = index.with_suffix(".tmp")

    with staged_part.open("wb") as stream:
        np.savez(stream, clip_ids=ids, cache_keys=keys, features=values)
    items = [
        {"clip_id": clip_id, "cache_key": key, "row": i, "part": part.name,
         "seconds": seconds / len(ids), "session": TAG}
        for i, (clip_id, key) in enumerate(zip(ids, keys))
    ]
    staged_index.write_text("".join(json.dumps(item) + "\n" for item in items))
    staged_part.replace(part)
    staged_index.replace(index)
    CACHE_INDEX.update({clip_id: (part, i) for i, clip_id in enumerate(ids)})


def load_layer20():
    by_part = defaultdict(list)
    for clip_id in REQUIRED:
        part, index = CACHE_INDEX[clip_id]
        by_part[part].append((clip_id, index))
    pooled = {}
    for part, entries in by_part.items():
        with np.load(part, allow_pickle=False) as archive:
            values = archive["features"]
            ids, keys = archive["clip_ids"], archive["cache_keys"]
            if values.shape != (len(ids), len(PROVENANCE["layer_names"]), WIDTH):
                raise ValueError(f"Incorrect pooled feature shape in {part}.")
            for clip_id, index in entries:
                if ids[index] != clip_id or keys[index] != CACHE_KEYS[clip_id]:
                    raise ValueError(f"Cache index mismatch for {clip_id} in {part}.")
                pooled[clip_id] = values[index, TAP_INDEX].astype(np.float32, copy=True)
    if not all(np.isfinite(value).all() for value in pooled.values()):
        raise ValueError("Layer 20 cache contains nonfinite values.")
    return pooled


discover_cache()
cache_report()


## Encode clips missing from the cache

In [ ]:
DATA_REVISIONS = {
    "UBC-NLP/NADI_2026_ADI20_micro": "c8da454c9f474bb075d3fe608940ee29e8ccfa43",
    "UBC-NLP/Casablanca": "8951b1b88e28c1107142ced57967b8d16350951d",
}


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def parquet_waves(rows, deadline, token):
    import pyarrow.parquet as pq
    import soundfile as sf
    from huggingface_hub import HfFileSystem, hf_hub_download

    hub = HfFileSystem(token=token)
    shards = defaultdict(list)
    for row in rows:
        dataset, revision, shard = (row[k] for k in ("source_dataset", "source_revision", "source_shard"))
        if DATA_REVISIONS.get(dataset) != revision:
            raise ValueError(f"Unexpected dataset revision for {row['clip_id']}.")
        if dataset.endswith("/Casablanca") and "validation" not in shard:
            raise ValueError(f"Unexpected Casablanca shard: {shard}.")
        shards[dataset, revision, shard].append(row)

    for (dataset, revision, shard), members in sorted(shards.items()):
        if time.monotonic() >= deadline:
            raise TimeoutError("Extraction time limit reached. The completed cache parts can be reused.")
        print(f"{shard} | {len(members):,} missing clips", flush=True)
        pending = {int(row["source_row"]): row for row in members}
        if len(pending) != len(members):
            raise ValueError(f"Duplicate source positions in {shard}.")
        with TemporaryDirectory(prefix="adi20-parquet-") as temp:
            if dataset.endswith("/Casablanca"):
                local = hf_hub_download(dataset, shard, revision=revision, repo_type="dataset",
                                        token=token, local_dir=temp)
                stream = open(local, "rb")
            else:
                stream = hub.open(f"datasets/{dataset}@{revision}/{shard}", "rb", block_size=8 * 1024**2)

            with stream:
                parquet = pq.ParquetFile(stream)
                start = 0
                for group in range(parquet.num_row_groups):
                    end = start + parquet.metadata.row_group(group).num_rows
                    needed = sorted(i for i in pending if start <= i < end)
                    cursor = start
                    start = end
                    if not needed:
                        continue
                    for batch in parquet.iter_batches(row_groups=[group], columns=["audio"], batch_size=32):
                        if time.monotonic() >= deadline:
                            raise TimeoutError("Extraction time limit reached. The completed cache parts can be reused.")
                        batch_end = cursor + len(batch)
                        block = []
                        for position in needed:
                            if not cursor <= position < batch_end:
                                continue
                            row = pending.pop(position)
                            payload = batch.column(0).field("bytes")[position - cursor].as_py()
                            if not payload or hashlib.sha256(payload).hexdigest() != row["audio_sha256"]:
                                raise ValueError(f"Audio hash mismatch for {row['clip_id']}.")
                            wave, rate = sf.read(io.BytesIO(payload), dtype="float32", always_2d=True)
                            block.append((row, ae.prepare_waveform(wave.mean(1), rate, FEATURE_CONFIG)))
                        if block:
                            yield block
                        cursor = batch_end
                        if cursor > needed[-1]:
                            break
                    if not pending:
                        break
            if pending:
                raise ValueError(f"Missing source rows in {shard}.")


def encode_missing(rows):
    if ENCODER_DEVICE.type != "cuda":
        raise RuntimeError("Enable a Kaggle GPU to encode missing clips, or attach a complete cache.")
    install_missing({"pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub", "hyperpyyaml": "hyperpyyaml",
                     "sentencepiece": "sentencepiece", "joblib": "joblib", "tqdm": "tqdm"})
    from importlib.metadata import PackageNotFoundError, version

    try:
        speechbrain_version = version("speechbrain")
    except PackageNotFoundError:
        speechbrain_version = ""
    if speechbrain_version != "1.1.2":
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "speechbrain==1.1.2"],
                       check=True)
    token = hf_token()
    if token:
        os.environ["HF_TOKEN"] = token
    encoder = ae.AcousticEncoder.from_pretrained(FEATURE_CONFIG, device=str(ENCODER_DEVICE), cache_dir="/tmp/adi-hf")
    if json.loads(json.dumps(encoder.provenance())) != PROVENANCE:
        raise ValueError("Loaded encoder does not match the original cache provenance.")
    ACTIVE_CACHE.mkdir(parents=True, exist_ok=True)
    (ACTIVE_CACHE / "config.json").write_text(json.dumps(CACHE_SPEC))
    deadline = time.monotonic() + MAX_EXTRACTION_HOURS * 3600

    @torch.inference_mode()
    def pooled_batch(waves):
        hidden, mask, _ = encoder.encode(waves)
        return encoder.pool(hidden, mask).cpu().numpy()

    for block in parquet_waves(rows, deadline, token):
        records, waves = zip(*block)
        tick = time.monotonic()
        arrays = []
        for start in range(0, len(waves), ENCODE_BATCH):
            group = list(waves[start:start + ENCODE_BATCH])
            try:
                values = pooled_batch(group)
            except torch.cuda.OutOfMemoryError:
                if len(group) == 1:
                    raise
                with torch.cuda.device(ENCODER_DEVICE):
                    torch.cuda.empty_cache()
                values = np.concatenate([pooled_batch([wave]) for wave in group])
            arrays.append(values)
        values = np.concatenate(arrays)
        if not np.isfinite(values).all():
            raise ValueError("AraBEST returned nonfinite features.")
        write_cache(records, values, time.monotonic() - tick)


## Define the matched E11 head

In [ ]:
class VariationalDialectHead(nn.Module):
    def __init__(self):
        super().__init__()
        # Keep the original module construction order so each seed has the same initialization.
        self.project = nn.Sequential(nn.Linear(WIDTH, 256), nn.GELU(), nn.Linear(256, LATENT_DIM))
        self.classifier = nn.Linear(LATENT_DIM, len(LABELS))
        self.logvar = nn.Linear(256, LATENT_DIM)

    def posterior(self, x):
        hidden = self.project[1](self.project[0](x))
        return self.project[2](hidden), self.logvar(hidden).clamp(-12.0, 8.0)


def diagonal_kl(mean, logvar):
    return 0.5 * (mean.square() + logvar.exp() - logvar - 1).sum(dim=-1)


def standardizer(raw):
    total = np.zeros(raw.shape[1], np.float64)
    squares = np.zeros_like(total)
    for start in range(0, len(raw), 256):
        block = raw[start:start + 256].astype(np.float64)
        total += block.sum(0)
        squares += np.square(block).sum(0)
    mean = total / len(raw)
    std = np.sqrt(np.maximum(squares / len(raw) - mean ** 2, 0))
    return mean.astype(np.float32), np.maximum(std.astype(np.float32), CONFIG["preprocessing"]["std_floor"])


def mini_batches(n, seed):
    _, shuffle_seed = clf.run_seeds(seed)
    generator = torch.Generator().manual_seed(shuffle_seed)
    while True:
        yield from torch.randperm(n, generator=generator).split(BATCH_SIZE)


def train_head(x, y, seed, beta):
    torch.manual_seed(seed)
    model = VariationalDialectHead().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    noise = torch.Generator(device=DEVICE).manual_seed(seed + 111)
    batches = mini_batches(len(y), seed)
    x = torch.as_tensor(x, device=DEVICE)
    y = torch.as_tensor(y, device=DEVICE)
    model.train()
    for _ in range(UPDATES):
        positions = next(batches).to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        mean, logvar = model.posterior(x.index_select(0, positions))
        epsilon = torch.randn(mean.shape, device=mean.device, dtype=mean.dtype, generator=noise)
        logits = model.classifier(mean + logvar.mul(0.5).exp() * epsilon)
        loss = F.cross_entropy(logits, y.index_select(0, positions)) + beta * diagonal_kl(mean, logvar).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["optimization"]["gradient_clip_norm"])
        optimizer.step()
    return model.eval()

## Fit the regularization controls and export the paired results

In [ ]:
if PREFLIGHT_ONLY:
    print("Preflight complete. Set PREFLIGHT_ONLY = False to fit the heads.")
else:
    missing = [row for clip_id, row in REQUIRED.items() if clip_id not in CACHE_INDEX]
    if missing and ENCODE_MISSING:
        encode_missing(missing)
        cache_report()

    from sklearn.metrics import precision_recall_fscore_support

    control_dir = OUT
    control_dir.mkdir(exist_ok=True)
    (control_dir / "predictions").mkdir(exist_ok=True)
    (control_dir / "heads").mkdir(exist_ok=True)

    BETA = 0.01
    # (coefficient of mean term, coefficient of variance term, AdamW decay)
    arms = {
        "beta0": (0.0, 0.0, 0.01),
        "mean_only": (BETA, 0.0, 0.01),
        "variance_only": (0.0, BETA, 0.01),
        "vib": (BETA, BETA, 0.01),
        "vib_shared_variance": (BETA, BETA, 0.01),
        "decay_0.001": (0.0, 0.0, 0.001),
        "decay_0.1": (0.0, 0.0, 0.1),
    }
    decay_candidates = ("decay_0.001", "beta0", "decay_0.1")

    # Fixed-input objective check against the original notebook's KL expression.
    def kl_parts(mean, logvar):
        mean_cost = 0.5 * mean.square().sum(dim=-1)
        variance_cost = 0.5 * (torch.expm1(logvar) - logvar).sum(dim=-1)
        return mean_cost, variance_cost

    probe_mean = torch.tensor([[0.0, 0.5], [-0.25, 1.0]])
    probe_logvar = torch.tensor([[0.0, 0.7], [-0.8, 1.5]])
    torch.testing.assert_close(
        sum(kl_parts(probe_mean, probe_logvar)), diagonal_kl(probe_mean, probe_logvar)
    )

    class SharedVarianceDialectHead(VariationalDialectHead):
        def posterior(self, x):
            mean, _ = super().posterior(x)
            logvar = self.logvar.bias.clamp(-12.0, 8.0).expand_as(mean)
            return mean, logvar

    # Exact historical endpoints use the original training function.
    def fit_control(x, y, seed, arm):
        if arm in ("beta0", "vib"):
            return train_head(x, y, seed, 0.0 if arm == "beta0" else BETA)

        mean_weight, variance_weight, decay = arms[arm]
        torch.manual_seed(seed)
        head = SharedVarianceDialectHead if arm == "vib_shared_variance" else VariationalDialectHead
        model = head().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=decay)
        noise = torch.Generator(device=DEVICE).manual_seed(seed + 111)
        x = torch.as_tensor(x, device=DEVICE)
        y = torch.as_tensor(y, device=DEVICE)
        batches = mini_batches(len(y), seed)
        model.train()
        for _ in range(UPDATES):
            positions = next(batches).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            mean, logvar = model.posterior(x.index_select(0, positions))
            eps = torch.randn(mean.shape, device=DEVICE, dtype=mean.dtype, generator=noise)
            logits = model.classifier(mean + logvar.mul(0.5).exp() * eps)
            mean_cost, variance_cost = kl_parts(mean, logvar)
            loss = (F.cross_entropy(logits, y.index_select(0, positions))
                    + mean_weight * mean_cost.mean() + variance_weight * variance_cost.mean())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["optimization"]["gradient_clip_norm"])
            optimizer.step()
        return model.eval()

    @torch.inference_mode()
    def score_features(model, features):
        predictions, parts = [], []
        for start in range(0, len(features), 512):
            block = torch.as_tensor(features[start:start + 512], device=DEVICE)
            mean, logvar = model.posterior(block)
            logits = model.classifier(mean)
            costs = torch.stack(kl_parts(mean, logvar), dim=1)
            if not (torch.isfinite(logits).all() and torch.isfinite(costs).all()):
                raise FloatingPointError("Invalid logits or KL terms.")
            predictions.append(logits.argmax(dim=1).cpu().numpy())
            parts.append(costs.cpu().numpy())
        return np.concatenate(predictions), np.concatenate(parts).mean(axis=0)

    if any(clip_id not in CACHE_INDEX for clip_id in REQUIRED):
        raise RuntimeError("Attach the original AraBEST pooled cache so every usable clip has a feature row.")
    if LATENT_DIM != 128 or TAP != "conformer_20" or UPDATES != 1000 or BATCH_SIZE != 256:
        raise ValueError("Use the original 128-d Layer 20 settings, 1000 updates and batch size 256.")

    protocol = {
        "arms_mean_beta_variance_beta_adamw_decay": arms,
        "tuning_rule": "Highest mean ADI20_dev200 macro-F1 across seeds. Ties prefer decay nearest 0.01.",
        "tap": TAP, "encoder": PROVENANCE, "source_commit": COMMIT,
        "training_draws": "fresh_balanced" if DRAW_NEW_COHORTS else "historical",
        "train_ids_sha256": TRAIN_DIGESTS, "eval_ids_sha256": ORIGINAL_EVAL_DIGESTS,
        "seeds": list(TRAIN_BY_SEED), "latent_dim": LATENT_DIM, "updates": UPDATES,
        "learning_rate": 0.001, "batch_size": BATCH_SIZE, "torch": str(torch.__version__),
        "device": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else str(DEVICE),
        "inference": "Posterior mean, unrestricted 20-way argmax. Casablanca macro-F1 uses 8 reference labels.",
        "evaluation_role": {"ADI20_dev200": "source development", "Casablanca_full6819": "external development"},
    }
    (control_dir / "protocol.json").write_text(json.dumps(protocol, indent=2))

    pooled = load_layer20()
    valid = {split: np.asarray([i for i, row in enumerate(rows) if usable(row)], dtype=int)
             for split, rows in EVAL.items()}
    raw_eval = {split: np.stack([pooled[EVAL[split][i]["clip_id"]] for i in indexes])
                for split, indexes in valid.items()}
    metrics, dialects = [], []

    for seed, rows in TRAIN_BY_SEED.items():
        train_raw = np.stack([pooled[row["clip_id"]] for row in rows])
        mean, std = standardizer(train_raw)
        x, y = (train_raw - mean) / std, targets(rows)
        eval_x = {split: (values - mean) / std for split, values in raw_eval.items()}
        for arm in arms:
            print(seed, arm, flush=True)
            model = fit_control(x, y, seed, arm)
            name = f"{arm}_seed{seed}"
            torch.save({
                "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "mean": torch.from_numpy(mean), "std": torch.from_numpy(std),
                "seed": seed, "arm": arm, "labels": LABELS,
            }, control_dir / "heads" / f"{name}.pt")
            for split, cohort in EVAL.items():
                pred, (mean_kl, variance_kl) = score_features(model, eval_x[split])
                full = np.full(len(cohort), -1, dtype=np.int64)
                full[valid[split]] = pred
                truth = targets(cohort)
                labels = list(range(len(LABELS))) if split == "ADI20_dev200" else SHARED
                p, r, f1, support = precision_recall_fscore_support(
                    truth, full, labels=labels, zero_division=0
                )
                row = dict(seed=seed, arm=arm, split=split)
                metrics.append(dict(
                    row, macro_f1=float(f1.mean()), accuracy=float((full == truth).mean()),
                    kl_mean_nats=float(mean_kl), kl_variance_nats=float(variance_kl),
                    n=len(cohort), scored=len(pred), coverage=len(pred) / len(cohort),
                ))
                dialects.extend(dict(row, dialect=LABELS[i], precision=pp, recall=rr, f1=ff, support=int(nn))
                                for i, pp, rr, ff, nn in zip(labels, p, r, f1, support))
                pd.DataFrame({
                    "clip_id": [row["clip_id"] for row in cohort],
                    "reference": [row["label"] for row in cohort],
                    "prediction": [LABELS[i] if i >= 0 else "FAILED" for i in full],
                }).to_csv(control_dir / "predictions" / f"{name}_{split}.csv.gz", index=False)
            del model
        pd.DataFrame(metrics).to_csv(control_dir / "metrics_per_seed.csv", index=False)
        pd.DataFrame(dialects).to_csv(control_dir / "per_dialect.csv", index=False)

    metrics = pd.DataFrame(metrics)
    source = metrics.loc[metrics.split.eq("ADI20_dev200")].groupby("arm")["macro_f1"].mean()
    decay_arm = max(decay_candidates, key=lambda arm: (source[arm], -abs(np.log10(arms[arm][2] / 0.01))))
    (control_dir / "source_selection.json").write_text(json.dumps({
        "decay_arm": decay_arm, "source_mean_macro_f1": source.to_dict(),
    }, indent=2))

    wide = metrics.pivot(index=["split", "seed"], columns="arm", values="macro_f1")
    contrasts = pd.DataFrame({
        "full_vs_beta0_pp": 100 * (wide["vib"] - wide["beta0"]),
        "mean_vs_beta0_pp": 100 * (wide["mean_only"] - wide["beta0"]),
        "variance_vs_beta0_pp": 100 * (wide["variance_only"] - wide["beta0"]),
        "full_vs_shared_variance_pp": 100 * (wide["vib"] - wide["vib_shared_variance"]),
        "full_vs_mean_pp": 100 * (wide["vib"] - wide["mean_only"]),
        "full_vs_tuned_decay_pp": 100 * (wide["vib"] - wide[decay_arm]),
        "mean_variance_interaction_pp": 100 * (
            wide["vib"] - wide["mean_only"] - wide["variance_only"] + wide["beta0"]
        ),
    })
    contrasts.to_csv(control_dir / "paired_contrasts.csv")
    summary = contrasts.groupby(level="split").agg(["mean", "std"])
    summary.to_csv(control_dir / "contrast_summary.csv")
    display(metrics.groupby(["split", "arm"])[["macro_f1", "kl_mean_nats", "kl_variance_nats"]].mean().round(4))
    display(summary.round(2))
    archive = Path(shutil.make_archive(str(control_dir), "zip", root_dir=control_dir))
    print(f"Results ready: {archive}")
    display(FileLink(os.path.relpath(archive)))